In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
# =========================================
# 🔥 CONFIG
# =========================================

N_TRIALS = 5
N_TRIALS_META = 5
N_TRIALS_BLEND = 5
PSEUDO_THRESHOLD = 0.85
N_SPLITS = 5
RANDOM_STATE = 7

DATA_PATH = "/kaggle/input/competitions/playground-series-s6e4/"

# =========================================
# 🔥 IMPORTS
# =========================================

import numpy as np
import pandas as pd
import optuna
import shap

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

print("✅ Imports done")

# =========================================
# 🔥 LOAD DATA
# =========================================

train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

target_col = train.columns[-1]

y = train[target_col]
X = train.drop(columns=[target_col])
X_test = test.copy()

# =========================================
# 🔥 FEATURE ENGINEERING (STRONG)
# =========================================

def create_features(df):
    df = df.copy()

    df["evapo"] = df["Temperature_C"] * (1 - df["Humidity"]/100)
    df["water_deficit"] = df["evapo"] - df["Rainfall_mm"]
    df["stress"] = df["Temperature_C"] / (df["Soil_Moisture"] + 1)

    df["moisture_rain_ratio"] = df["Soil_Moisture"] / (df["Rainfall_mm"] + 1)
    df["rain_temp_ratio"] = df["Rainfall_mm"] / (df["Temperature_C"] + 1)

    df["temp_humidity"] = df["Temperature_C"] * df["Humidity"]
    df["rain_moisture"] = df["Rainfall_mm"] * df["Soil_Moisture"]

    df["temp_squared"] = df["Temperature_C"] ** 2
    df["rain_log"] = np.log1p(df["Rainfall_mm"])

    df["wind_sun"] = df["Wind_Speed_kmh"] * df["Sunlight_Hours"]
    df["temp_sun"] = df["Temperature_C"] * df["Sunlight_Hours"]

    df["crop_stage"] = df["Crop_Type"].astype(str) + "_" + df["Crop_Growth_Stage"].astype(str)
    df["region_season"] = df["Region"].astype(str) + "_" + df["Season"].astype(str)

    df["crop_water"] = df["Crop_Type"].astype(str) + "_" + df["Water_Source"].astype(str)
    df["soil_crop"] = df["Soil_Type"].astype(str) + "_" + df["Crop_Type"].astype(str)

    return df

X = create_features(X)
X_test = create_features(X_test)

# =========================================
# 🔥 ENCODING
# =========================================

for col in X.select_dtypes("object").columns:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

cat_cols = X.select_dtypes("category").columns.tolist()

le = LabelEncoder()
y = le.fit_transform(y)
num_classes = len(np.unique(y))

# =========================================
# 🔥 SHAP FEATURE SELECTION
# =========================================

print("🚀 SHAP selection...")

sample_idx = np.random.choice(len(X), min(2000, len(X)), replace=False)
X_sample = X.iloc[sample_idx].copy()
y_sample = y[sample_idx]

for col in X_sample.select_dtypes("category").columns:
    X_sample[col] = X_sample[col].cat.codes

temp_model = xgb.XGBClassifier(n_estimators=120, tree_method="hist")
temp_model.fit(X_sample, y_sample)

explainer = shap.TreeExplainer(temp_model)
shap_vals = explainer.shap_values(X_sample, approximate=True)

if isinstance(shap_vals, list):
    shap_vals = np.stack(shap_vals, axis=-1)

importance = np.abs(shap_vals).mean(axis=(0,2))
keep_cols = X.columns[importance > np.percentile(importance, 20)]

X = X[keep_cols]
X_test = X_test[keep_cols]

cat_cols = [c for c in cat_cols if c in X.columns]

print("✅ Features:", len(keep_cols))

# =========================================
# 🔥 XGB SAFE DATA
# =========================================

X_xgb = X.copy()
X_test_xgb = X_test.copy()

for col in X_xgb.select_dtypes("category").columns:
    X_xgb[col] = X_xgb[col].cat.codes
    X_test_xgb[col] = X_test_xgb[col].cat.codes

# =========================================
# 🔥 CV
# =========================================

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# =========================================
# 🔥 OPTUNA BASE MODELS
# =========================================

def xgb_obj(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": num_classes,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_estimators": 400,
        "tree_method": "hist"
    }
    scores=[]
    for tr,val in skf.split(X_xgb,y):
        m=xgb.XGBClassifier(**params)
        m.fit(X_xgb.iloc[tr],y[tr])
        scores.append(balanced_accuracy_score(y[val],m.predict(X_xgb.iloc[val])))
    return np.mean(scores)

def lgb_obj(trial):
    params={
        "objective":"multiclass",
        "num_class":num_classes,
        "learning_rate":trial.suggest_float("learning_rate",0.01,0.1),
        "num_leaves":trial.suggest_int("num_leaves",20,100),
        "n_estimators":400
    }
    scores=[]
    for tr,val in skf.split(X,y):
        m=lgb.LGBMClassifier(**params)
        m.fit(X.iloc[tr],y[tr])
        scores.append(balanced_accuracy_score(y[val],m.predict(X.iloc[val])))
    return np.mean(scores)

def cat_obj(trial):
    params={
        "iterations":400,
        "learning_rate":trial.suggest_float("learning_rate",0.01,0.1),
        "depth":trial.suggest_int("depth",3,8),
        "verbose":0
    }
    scores=[]
    for tr,val in skf.split(X,y):
        m=CatBoostClassifier(**params)
        m.fit(X.iloc[tr],y[tr],cat_features=cat_cols)
        scores.append(balanced_accuracy_score(y[val],m.predict(X.iloc[val])))
    return np.mean(scores)

def et_obj(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators",200,500),
        "max_depth": trial.suggest_int("max_depth",6,20),
        "min_samples_split": trial.suggest_int("min_samples_split",2,10),
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }

    scores=[]
    for tr,val in skf.split(X_xgb,y):
        m = ExtraTreesClassifier(**params)
        m.fit(X_xgb.iloc[tr],y[tr])
        scores.append(balanced_accuracy_score(y[val],m.predict(X_xgb.iloc[val])))
    return np.mean(scores)

print("🚀 tuning models")



study_xgb=optuna.create_study(direction="maximize")
study_xgb.optimize(xgb_obj,n_trials=N_TRIALS)
print("XGB Optuna DONE ✅")

study_lgb=optuna.create_study(direction="maximize")
study_lgb.optimize(lgb_obj,n_trials=N_TRIALS)
print("LGB Optuna DONE ✅✅")

# study_cat=optuna.create_study(direction="maximize")
# study_cat.optimize(cat_obj,n_trials=N_TRIALS)

study_et = optuna.create_study(direction="maximize")
study_et.optimize(et_obj, n_trials=N_TRIALS)
print("ET Optuna DONE ✅✅✅")

# =========================================
# 🔥 OOF STACKING
# =========================================

oof_xgb=np.zeros((len(X),num_classes))
oof_lgb=np.zeros((len(X),num_classes))
# oof_cat=np.zeros((len(X),num_classes))
oof_et  = np.zeros((len(X),num_classes))

for tr,val in skf.split(X,y):

    xgb_m=xgb.XGBClassifier(**study_xgb.best_params,n_estimators=400)
    lgb_m=lgb.LGBMClassifier(**study_lgb.best_params,n_estimators=400)
    # cat_m=CatBoostClassifier(**study_cat.best_params,iterations=400,verbose=0)
    et_m  = ExtraTreesClassifier(**study_et.best_params)

    xgb_m.fit(X_xgb.iloc[tr],y[tr])
    lgb_m.fit(X.iloc[tr],y[tr])
    # cat_m.fit(X.iloc[tr],y[tr],cat_features=cat_cols)
    et_m.fit(X_xgb.iloc[tr],y[tr])

    oof_xgb[val]=xgb_m.predict_proba(X_xgb.iloc[val])
    oof_lgb[val]=lgb_m.predict_proba(X.iloc[val])
    # oof_cat[val]=cat_m.predict_proba(X.iloc[val])
    oof_et[val]=et_m.predict_proba(X_xgb.iloc[val])

# X_meta=np.hstack([oof_xgb,oof_lgb,oof_cat])
X_meta = np.hstack([oof_xgb,oof_lgb,oof_et])

# =========================================
# 🔥 META OPTUNA
# =========================================

def meta_obj(trial):
    C=trial.suggest_float("C",0.01,10)
    m=LogisticRegression(C=C,max_iter=1000)
    scores=[]
    for tr,val in skf.split(X_meta,y):
        m.fit(X_meta[tr],y[tr])
        scores.append(balanced_accuracy_score(y[val],m.predict(X_meta[val])))
    return np.mean(scores)

study_meta=optuna.create_study(direction="maximize")
study_meta.optimize(meta_obj,n_trials=N_TRIALS_META)
print("META Optuna DONE ✅✅✅✅")

meta=LogisticRegression(**study_meta.best_params,max_iter=1000)
meta.fit(X_meta,y)

# =========================================
# 🔥 FINAL TRAIN
# =========================================

xgb_f=xgb.XGBClassifier(**study_xgb.best_params,n_estimators=600)
lgb_f=lgb.LGBMClassifier(**study_lgb.best_params,n_estimators=600)
# cat_f=CatBoostClassifier(**study_cat.best_params,iterations=600,verbose=0)
et_f  = ExtraTreesClassifier(**study_et.best_params)

xgb_f.fit(X_xgb,y)
lgb_f.fit(X,y)
# cat_f.fit(X,y,cat_features=cat_cols)
et_f.fit(X_xgb,y)

test_xgb=xgb_f.predict_proba(X_test_xgb)
test_lgb=lgb_f.predict_proba(X_test)
# test_cat=cat_f.predict_proba(X_test)
test_et  = et_f.predict_proba(X_test_xgb)

# stack_test=np.hstack([test_xgb,test_lgb,test_cat])
stack_test = np.hstack([test_xgb,test_lgb,test_et])
stack_preds=meta.predict_proba(stack_test)

# =========================================
# 🔥 BLENDING OPTUNA
# =========================================

def blend_obj(trial):
    w1=trial.suggest_float("xgb",0,1)
    w2=trial.suggest_float("lgb",0,1)
    # w3=trial.suggest_float("cat",0,1)
    w3=trial.suggest_float("et",0,1)
    w4=trial.suggest_float("meta",0,1)

    wsum=w1+w2+w3+w4
    w1,w2,w3,w4=[w/wsum for w in [w1,w2,w3,w4]]

    # blend=w1*oof_xgb+w2*oof_lgb+w3*oof_cat+w4*meta.predict_proba(X_meta)
    blend=w1*oof_xgb+w2*oof_lgb+w3*oof_et+w4*meta.predict_proba(X_meta)
    
    preds=np.argmax(blend,axis=1)
    return balanced_accuracy_score(y,preds)

study_blend=optuna.create_study(direction="maximize")
study_blend.optimize(blend_obj,n_trials=N_TRIALS_BLEND)
print("BLEND Optuna DONE ✅✅✅✅✅")

w=study_blend.best_params
wsum=sum(w.values())
w={k:v/wsum for k,v in w.items()}

final_probs=(w["xgb"]*test_xgb+
             w["lgb"]*test_lgb+
             # w["cat"]*test_cat+
             w["et"]*test_et+
             w["meta"]*stack_preds)

# =========================================
# 🔥 PSEUDO LABELING (SAFE)
# =========================================

conf=np.max(final_probs,axis=1)
mask=conf>PSEUDO_THRESHOLD

if mask.sum()>0:
    print("Pseudo samples:",mask.sum())

    X_aug=pd.concat([X,X_test[mask]])
    y_aug=np.concatenate([y,np.argmax(final_probs[mask],axis=1)])

    xgb_f.fit(pd.concat([X_xgb,X_test_xgb[mask]]),y_aug)
    test_xgb=xgb_f.predict_proba(X_test_xgb)

# =========================================
# 🔥 FINAL OUTPUT
# =========================================

final_probs=(w["xgb"]*test_xgb+
             w["lgb"]*test_lgb+
             # w["cat"]*test_cat+
             w["et"]*test_et+
             w["meta"]*stack_preds)

final_preds=le.inverse_transform(np.argmax(final_probs,axis=1))

sub=pd.read_csv(DATA_PATH+"sample_submission.csv")
sub[target_col]=final_preds
sub.to_csv("submission.csv",index=False)

print("🚀 FINAL SUBMISSION READY")

✅ Imports done
🚀 SHAP selection...


[I 2026-04-22 10:13:36,151] A new study created in memory with name: no-name-60804f42-2a86-4d49-bfdc-21a77736eb67


✅ Features: 28
🚀 tuning models


[I 2026-04-22 10:16:57,686] Trial 0 finished with value: 0.9618367271191864 and parameters: {'learning_rate': 0.040325085703753295, 'max_depth': 6, 'subsample': 0.780124076146436, 'colsample_bytree': 0.9613241985629405}. Best is trial 0 with value: 0.9618367271191864.
[I 2026-04-22 10:19:44,679] Trial 1 finished with value: 0.9619527698724009 and parameters: {'learning_rate': 0.08166693683910461, 'max_depth': 5, 'subsample': 0.870742629742431, 'colsample_bytree': 0.9793529127607226}. Best is trial 1 with value: 0.9619527698724009.
[I 2026-04-22 10:22:29,520] Trial 2 finished with value: 0.961938620498844 and parameters: {'learning_rate': 0.08709161163124572, 'max_depth': 5, 'subsample': 0.7910434348250166, 'colsample_bytree': 0.617155068556407}. Best is trial 1 with value: 0.9619527698724009.
[I 2026-04-22 10:26:00,421] Trial 3 finished with value: 0.961989857578623 and parameters: {'learning_rate': 0.07438582232988215, 'max_depth': 7, 'subsample': 0.7842827396790545, 'colsample_bytree

XGB Optuna DONE ✅
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.074558 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5280
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017146 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5278
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968943
[LightGBM] [In

...

ET Optuna DONE ✅✅✅
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016788 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5280
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400721
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968948
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016874 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5278
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532443
[Li

[I 2026-04-22 12:06:48,963] A new study created in memory with name: no-name-8fc7f0c1-a207-4cc5-9b92-b6a68de8a134
[I 2026-04-22 12:07:00,945] Trial 0 finished with value: 0.9620623769308922 and parameters: {'C': 5.708742762775321}. Best is trial 0 with value: 0.9620623769308922.
[I 2026-04-22 12:07:12,569] Trial 1 finished with value: 0.9620674608144528 and parameters: {'C': 3.3922836608783506}. Best is trial 1 with value: 0.9620674608144528.
[I 2026-04-22 12:07:23,924] Trial 2 finished with value: 0.9620637711935016 and parameters: {'C': 6.852600941187513}. Best is trial 1 with value: 0.9620674608144528.
[I 2026-04-22 12:07:35,625] Trial 3 finished with value: 0.9620637711935016 and parameters: {'C': 6.780544649580116}. Best is trial 1 with value: 0.9620674608144528.
[I 2026-04-22 12:07:46,830] Trial 4 finished with value: 0.9620838232357933 and parameters: {'C': 2.8607347736981406}. Best is trial 4 with value: 0.9620838232357933.


META Optuna DONE ✅✅✅✅
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022914 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5278
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 28
[LightGBM] [Info] Start training from score -3.400769
[LightGBM] [Info] Start training from score -0.532441
[LightGBM] [Info] Start training from score -0.968947


[I 2026-04-22 12:19:00,772] A new study created in memory with name: no-name-38f75213-79d2-46f8-b6d3-ee1875d680b0
[I 2026-04-22 12:19:00,897] Trial 0 finished with value: 0.9616797525441635 and parameters: {'xgb': 0.9654522892627696, 'lgb': 0.3247122373121679, 'et': 0.9711266542383853, 'meta': 0.6192590258199063}. Best is trial 0 with value: 0.9616797525441635.
[I 2026-04-22 12:19:01,009] Trial 1 finished with value: 0.9620834463146148 and parameters: {'xgb': 0.7013064633052108, 'lgb': 0.38811166238640393, 'et': 0.2563321138945379, 'meta': 0.5055190536208505}. Best is trial 1 with value: 0.9620834463146148.
[I 2026-04-22 12:19:01,125] Trial 2 finished with value: 0.9620881353174799 and parameters: {'xgb': 0.44105880244456375, 'lgb': 0.8794068486228438, 'et': 0.47407636053902313, 'meta': 0.716496134850271}. Best is trial 2 with value: 0.9620881353174799.
[I 2026-04-22 12:19:01,238] Trial 3 finished with value: 0.9611227526253799 and parameters: {'xgb': 0.09189146618581934, 'lgb': 0.2268

BLEND Optuna DONE ✅✅✅✅✅
Pseudo samples: 262361
🚀 FINAL SUBMISSION READY
